In [4]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader('astrocortex.txt')
documents = loader.load() # This returns a list of Document objects
astro_cortex_text = documents[0].page_content # Extract text if loaded from file

In [5]:
astro_cortex_text

"Project AstroCortex: An Overview\n\nAstroCortex is a groundbreaking initiative aimed at mapping the neural pathways of cosmic entities. Our primary objective is to understand how these beings perceive and interact with the universe's fundamental forces. The project integrates quantum physics, astrobiology, and computational neuroscience.\n\nMethodology involves deploying hyperspace probes equipped with neural scanners. These probes collect data signatures, which are then transmitted back to our Earth-based Deep Thought Cluster for analysis. Initial data suggests complex, non-linear thought patterns extremely different from terrestrial life. We are using advanced machine learning models to decode these patterns. Challenges include signal degradation over immense distances and the unique nature of the data, requiring novel analytical techniques.\n\nEthics are significant. Communication attempts are strictly non-invasive, adhering to the Prime Directive Protocols established in 2267. Ens

## Fixed-Size Chunking

The most straightforward approach is to split the text into chunks of a fixed character length. This method is simple but can sometimes awkwardly split sentences or ideas. We'll use LangChain's CharacterTextSplitter for this. An important parameter here is chunk_overlap, which defines how many characters from the end of one chunk are repeated at the beginning of the next. This overlap helps maintain context across chunk boundaries.

In [6]:
from langchain_text_splitters import CharacterTextSplitter

# Initialize the splitter
char_splitter = CharacterTextSplitter(
    separator="\n\n", # How to first try splitting text (e.g., by paragraph)
    chunk_size=250,    # Desired maximum chunk size (in characters)
    chunk_overlap=50,  # Number of characters to overlap between chunks
    length_function=len, # Function to measure chunk length (default is fine)
    is_separator_regex=False, # Treat separator as a literal string
)

# Split the text
fixed_chunks = char_splitter.split_text(astro_cortex_text)

# Let's examine the chunks
print(f"Original text length: {len(astro_cortex_text)}")
print(f"Number of chunks created: {len(fixed_chunks)}\n")

for i, chunk in enumerate(fixed_chunks):
    print(f"--- Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)
    print("-" * 20 + "\n")

Created a chunk of size 303, which is longer than the specified 250
Created a chunk of size 512, which is longer than the specified 250


Original text length: 1485
Number of chunks created: 4

--- Chunk 1 (Length: 32) ---
Project AstroCortex: An Overview
--------------------

--- Chunk 2 (Length: 303) ---
AstroCortex is a groundbreaking initiative aimed at mapping the neural pathways of cosmic entities. Our primary objective is to understand how these beings perceive and interact with the universe's fundamental forces. The project integrates quantum physics, astrobiology, and computational neuroscience.
--------------------

--- Chunk 3 (Length: 512) ---
Methodology involves deploying hyperspace probes equipped with neural scanners. These probes collect data signatures, which are then transmitted back to our Earth-based Deep Thought Cluster for analysis. Initial data suggests complex, non-linear thought patterns extremely different from terrestrial life. We are using advanced machine learning models to decode these patterns. Challenges include signal degradation over immense distances and the unique nature of the data, 

## Recursive Character Chunking

This method attempts to split text based on a prioritized list of separators. It starts with larger semantic units (like double newlines for paragraphs) and recursively splits smaller pieces if a chunk is still too large, using progressively smaller separators (single newline, space, etc.). This approach tends to keep related content together more effectively than simple fixed-size splitting.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the recursive splitter
recursive_splitter = RecursiveCharacterTextSplitter(
    # Try splitting first by paragraphs, then sentences, then words
    separators=["\n\n", "\n", " ", ""],
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

# Split the text
recursive_chunks = recursive_splitter.split_text(astro_cortex_text)

# Examine the results
print(f"Original text length: {len(astro_cortex_text)}")
print(f"Number of chunks created by recursive splitter: {len(recursive_chunks)}\n")

for i, chunk in enumerate(recursive_chunks):
    print(f"--- Recursive Chunk {i+1} (Length: {len(chunk)}) ---")
    print(chunk)
    print("-" * 20 + "\n")


Original text length: 1485
Number of chunks created by recursive splitter: 9

--- Recursive Chunk 1 (Length: 32) ---
Project AstroCortex: An Overview
--------------------

--- Recursive Chunk 2 (Length: 248) ---
AstroCortex is a groundbreaking initiative aimed at mapping the neural pathways of cosmic entities. Our primary objective is to understand how these beings perceive and interact with the universe's fundamental forces. The project integrates quantum
--------------------

--- Recursive Chunk 3 (Length: 93) ---
forces. The project integrates quantum physics, astrobiology, and computational neuroscience.
--------------------

--- Recursive Chunk 4 (Length: 245) ---
Methodology involves deploying hyperspace probes equipped with neural scanners. These probes collect data signatures, which are then transmitted back to our Earth-based Deep Thought Cluster for analysis. Initial data suggests complex, non-linear
--------------------

--- Recursive Chunk 5 (Length: 249) ---
Initial data s